# Notebook 01 — EDA NHAMCS

**Propósito:** Cargar los datos NHAMCS, construir el dataset procesado, y responder las preguntas descriptivas del EDA.  
**No contiene decisiones metodológicas** — esas están fijadas en el Notebook 00.  
**No entrena modelos** — solo números, distribuciones y visualizaciones.

**Preguntas que responde este notebook:**
1. ¿Cuántos registros hay por ESI y por año?
2. ¿Cuál es el base rate del outcome por ESI? ¿Cuántos casos tiene cada componente?
3. ¿Cuál es el missingness de las variables predictoras clave?
4. ¿Cuántas alertas generaría el Top 5%, 10%, 15% por ESI?
5. ¿Qué distribución tiene AGE por ESI? ¿Cuántos pacientes ≥65 hay?
6. ¿Cuál es la distribución de RFV1 por ESI?
7. ¿Cuál es la distribución de comorbilidades por ESI?
8. ¿Cuál es la distribución demográfica (RACERETH, SEX, PAYTYPER) por ESI?

**Output:** `data/processed/nhamcs_pooled_2016_2022.csv` y `reports/01_eda_nhamcs/`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
import json
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_DIR = Path(r'C:\Users\pablo\OneDrive\Desktop\triagegeist_submission')
NHAMCS_DIR  = PROJECT_DIR / 'NHAMCS'
DATA_DIR    = PROJECT_DIR / 'data' / 'processed'
REPORT_DIR  = PROJECT_DIR / 'reports' / '01_eda_nhamcs'
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2016, 2017, 2018, 2019, 2022]

YEAR_FILES = {
    2016: NHAMCS_DIR / '2016' / 'ed2016' / 'ed2016',
    2017: NHAMCS_DIR / '2017' / 'ED2017' / 'ED2017',
    2018: NHAMCS_DIR / '2018' / 'ED2018' / 'ED2018',
    2019: NHAMCS_DIR / '2019' / 'ED2019' / 'ed2019',
    2022: NHAMCS_DIR / '2022' / 'ed2022' / 'ed2022',
}

print('Verificando archivos NHAMCS:')
for yr, path in YEAR_FILES.items():
    size_mb = path.stat().st_size / 1_048_576 if path.exists() else 0
    status  = f'OK  ({size_mb:.1f} MB)' if path.exists() else 'FALTA'
    print(f'  {yr}: {status}')

print(f'\npandas {pd.__version__}  numpy {np.__version__}')

Verificando archivos NHAMCS:
  2016: OK  (45.6 MB)
  2017: OK  (39.1 MB)
  2018: OK  (47.6 MB)
  2019: OK  (44.3 MB)
  2022: OK  (36.4 MB)

pandas 2.3.0  numpy 2.3.0


---
## Sección 1 — Ingesta de datos raw (fixed-width ASCII)

Los archivos NHAMCS son fixed-width ASCII sin encabezados.  
Las posiciones de columna están verificadas contra `docs/doc22-ed-508.pdf` (CDC) y el notebook 11 de V1.  

**Diferencia con V1:** se agrega `POPCT` (saturación de O2) en posición `(63, 66)` — omitida en V1.  
**Ajuste 2022:** comorbilidades y outcomes tienen offset distinto; vitales son idénticos a 2016-2019.

In [2]:
# Columnas compartidas para todos los años (vitales de triage + demografía básica)
# Formato: (start_0indexed, end_exclusive, nombre)
# Verificado contra doc22-ed-508.pdf página 36 y SAS companion files 2016-2019

_SHARED = [
    ( 7, 11, 'WAITTIME'),
    (11, 15, 'LOV'),
    (15, 18, 'AGE'),
    (24, 25, 'SEX'),
    (31, 32, 'RACERETH'),
    (32, 34, 'ARREMS'),
    (45, 47, 'PAYTYPER'),
    (47, 51, 'TEMPF'),      # stored 10x: 981 = 98.1 F
    (51, 54, 'PULSE'),
    (54, 57, 'RESPR'),
    (57, 60, 'BPSYS'),
    (60, 63, 'BPDIAS'),
    (63, 66, 'POPCT'),      # SpO2 0-100%; omitida en V1, confirmada doc pág 36 y validada empíricamente
    (66, 68, 'IMMEDR'),     # ESI 1-5; solo como filtro, nunca predictor
    (68, 70, 'PAINSCALE'),
    (72, 77, 'RFV1'),
    (77, 82, 'RFV2'),
]

_COMORBID_PRE2022 = [
    (153, 154, 'ASTHMA'),
    (154, 155, 'CANCER'),
    (157, 158, 'COPD'),
    (158, 159, 'CHF'),
    (160, 161, 'DEPRN'),
    (161, 162, 'DIABTYP1'),
    (162, 163, 'DIABTYP2'),
    (164, 165, 'ESRD'),
    (166, 167, 'EDHIV'),
    (168, 169, 'HTN'),
    (169, 170, 'OBESITY'),
    (172, 173, 'SUBSTAB'),
]

_COMORBID_2022 = [
    (151, 152, 'ASTHMA'),   # 2 bytes antes en layout 2022
    (154, 155, 'CANCER'),
    (157, 158, 'COPD'),
    (158, 159, 'CHF'),
    (160, 161, 'DEPRN'),
    (161, 162, 'DIABTYP1'),
    (162, 163, 'DIABTYP2'),
    (164, 165, 'ESRD'),
    (166, 167, 'EDHIV'),
    (168, 169, 'HTN'),
    (169, 170, 'OBESITY'),
    (172, 173, 'SUBSTAB'),
]

_OUTCOMES_PRE2022 = [
    (454, 456, 'NUMMED'),
    (488, 489, 'LWBS'),
    (490, 491, 'LEFTAMA'),
    (492, 493, 'DIEDED'),
    (493, 494, 'TRANNH'),
    (495, 496, 'TRANOTH'),
    (496, 497, 'ADMITHOS'),
]

_OUTCOMES_2022 = [
    (456, 458, 'NUMMED'),   # +2 bytes en layout 2022
    (490, 491, 'LWBS'),
    (492, 493, 'LEFTAMA'),
    (494, 495, 'DIEDED'),
    (495, 496, 'TRANNH'),
    (497, 498, 'TRANOTH'),
    (498, 499, 'ADMITHOS'),
]

COLSPECS = {
    2016: _SHARED + _COMORBID_PRE2022 + _OUTCOMES_PRE2022,
    2017: _SHARED + _COMORBID_PRE2022 + _OUTCOMES_PRE2022,
    2018: _SHARED + _COMORBID_PRE2022 + _OUTCOMES_PRE2022,
    2019: _SHARED + _COMORBID_PRE2022 + _OUTCOMES_PRE2022,
    2022: _SHARED + _COMORBID_2022    + _OUTCOMES_2022,
}

for yr, specs in COLSPECS.items():
    print(f'{yr}: {len(specs)} columnas')

2016: 36 columnas
2017: 36 columnas
2018: 36 columnas
2019: 36 columnas
2022: 36 columnas


In [3]:
def load_year(year):
    spec_list = COLSPECS[year]
    colspecs  = [(s, e) for s, e, _ in spec_list]
    names     = [n          for _, _, n in spec_list]
    df = pd.read_fwf(
        YEAR_FILES[year],
        colspecs=colspecs,
        names=names,
        header=None,
        dtype=str,
        encoding='latin-1',
    )
    df['year'] = year
    return df

dfs_raw = {}
for yr in YEARS:
    print(f'Cargando {yr}...', end=' ')
    dfs_raw[yr] = load_year(yr)
    print(f'{len(dfs_raw[yr]):,} filas')

print(f'\nTotal bruto: {sum(len(d) for d in dfs_raw.values()):,} filas')

Cargando 2016... 19,467 filas
Cargando 2017... 

16,709 filas
Cargando 2018... 20,291 filas
Cargando 2019... 

19,481 filas
Cargando 2022... 16,025 filas

Total bruto: 91,973 filas


In [4]:
# Validación rápida de POPCT para confirmar que la posición es correcta
# SpO2 esperado: mediana ~98, rango típico 85-100
print('Validación POPCT (SpO2) por año:')
for yr, df in dfs_raw.items():
    popct = pd.to_numeric(df['POPCT'], errors='coerce')
    valid = popct[popct.between(50, 105)]
    print(f'  {yr}: N válido={len(valid):,}  mediana={valid.median():.0f}  p5={valid.quantile(0.05):.0f}  p95={valid.quantile(0.95):.0f}')

Validación POPCT (SpO2) por año:
  2016: N válido=17,874  mediana=98  p5=93  p95=100
  2017: N válido=15,653  mediana=98  p5=94  p95=100
  2018: N válido=18,495  mediana=98  p5=94  p95=100
  2019: N válido=18,156  mediana=98  p5=94  p95=100
  2022: N válido=14,993  mediana=98  p5=94  p95=100


In [5]:
NUMERIC_COLS = [
    'WAITTIME', 'LOV', 'AGE', 'SEX', 'RACERETH', 'ARREMS', 'PAYTYPER',
    'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'IMMEDR',
    'PAINSCALE', 'RFV1', 'RFV2', 'NUMMED',
    'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2',
    'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB',
    'LWBS', 'LEFTAMA', 'DIEDED', 'TRANNH', 'TRANOTH', 'ADMITHOS',
]

def clean_year(df):
    df = df.copy()
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # TEMPF: stored 10x (981 = 98.1 F); fuera de rango → NaN
    df['temp_f'] = df['TEMPF'].where(df['TEMPF'].between(800, 1100)) / 10.0

    # Vitales: eliminar códigos de missing/invalid
    df['PULSE']     = df['PULSE'].where(df['PULSE'].between(1, 399))
    df['RESPR']     = df['RESPR'].where(df['RESPR'].between(1, 98))
    df['BPSYS']     = df['BPSYS'].where(df['BPSYS'].between(1, 998))
    df['BPDIAS']    = df['BPDIAS'].where(df['BPDIAS'].between(1, 299))
    df['POPCT']     = df['POPCT'].where(df['POPCT'].between(0, 100))
    df['PAINSCALE'] = df['PAINSCALE'].where(df['PAINSCALE'].between(0, 10))
    df['AGE']       = df['AGE'].where(df['AGE'].between(0, 120))

    # Outcome principal: hospitalización OR traslado OR muerte
    df['outcome'] = (
        (df['ADMITHOS'] == 1) |
        (df['TRANOTH']  == 1) |
        (df['DIEDED']   == 1)
    ).astype(int)

    # ESI: válido 1-5
    df['esi'] = df['IMMEDR'].where(df['IMMEDR'].between(1, 5))

    # Shock index
    df['shock_index'] = df['PULSE'] / df['BPSYS']
    df['shock_index'] = df['shock_index'].where(df['shock_index'].between(0.1, 5.0))

    # Flags auxiliares
    df['age_65plus'] = (df['AGE'] >= 65).astype('Int8')

    return df

dfs_clean = {yr: clean_year(df) for yr, df in dfs_raw.items()}

df_pool = pd.concat(list(dfs_clean.values()), ignore_index=True)
df_esi  = df_pool[df_pool['esi'].notna()].copy()
df_esi['esi'] = df_esi['esi'].astype(int)

# Scope del proyecto: ESI 3, 4, 5
df_main = df_esi[df_esi['esi'].isin([3, 4, 5])].copy()

print(f'Pool completo:       {len(df_pool):,}')
print(f'Con ESI 1-5:         {len(df_esi):,}')
print(f'Scope ESI 3-5:       {len(df_main):,}')

Pool completo:       91,973
Con ESI 1-5:         64,029
Scope ESI 3-5:       54,744


In [6]:
# Guardar dataset procesado
out_path = DATA_DIR / 'nhamcs_pooled_2016_2022.csv'
df_main.to_csv(out_path, index=False)
print(f'Guardado: {out_path}')
print(f'Shape: {df_main.shape}  ({df_main.shape[0]:,} filas x {df_main.shape[1]} columnas)')
print(f'Columnas: {list(df_main.columns)}')

Guardado: C:\Users\pablo\OneDrive\Desktop\triagegeist_submission\data\processed\nhamcs_pooled_2016_2022.csv
Shape: (54744, 42)  (54,744 filas x 42 columnas)
Columnas: ['WAITTIME', 'LOV', 'AGE', 'SEX', 'RACERETH', 'ARREMS', 'PAYTYPER', 'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'IMMEDR', 'PAINSCALE', 'RFV1', 'RFV2', 'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2', 'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB', 'NUMMED', 'LWBS', 'LEFTAMA', 'DIEDED', 'TRANNH', 'TRANOTH', 'ADMITHOS', 'year', 'temp_f', 'outcome', 'esi', 'shock_index', 'age_65plus']


---
## EDA 1 — Registros por ESI y por año

In [7]:
print('=== DISTRIBUCIÓN ESI POR AÑO (ESI 1-5) ===\n')

pivot = df_esi.groupby(['year', 'esi']).size().unstack(fill_value=0)
pivot.columns = [f'ESI_{int(c)}' for c in pivot.columns]
pivot['Total'] = pivot.sum(axis=1)
print(pivot.to_string())

print('\n--- Global ---')
vc  = df_esi['esi'].value_counts().sort_index()
pct = vc / len(df_esi) * 100
for e in range(1, 6):
    print(f'  ESI {e}: {vc.get(e,0):7,}  ({pct.get(e,0):.1f}%)')
print(f'  Total:  {len(df_esi):7,}')

print('\n--- Scope del proyecto: ESI 3-5 ---')
for e in [3, 4, 5]:
    n = (df_esi['esi'] == e).sum()
    print(f'  ESI {e}: {n:7,}  ({n/len(df_esi)*100:.1f}% del total con ESI)')

pivot.to_csv(REPORT_DIR / '01_esi_by_year.csv')

=== DISTRIBUCIÓN ESI POR AÑO (ESI 1-5) ===

      ESI_1  ESI_2  ESI_3  ESI_4  ESI_5  Total
year                                          
2016    104   1530   6526   4795    859  13814
2017    145   1539   5759   4100    667  12210
2018    180   1983   6954   4379    707  14203
2019    198   1872   6881   4082    562  13595
2022    139   1595   5340   2826    307  10207

--- Global ---
  ESI 1:     766  (1.2%)
  ESI 2:   8,519  (13.3%)
  ESI 3:  31,460  (49.1%)
  ESI 4:  20,182  (31.5%)
  ESI 5:   3,102  (4.8%)
  Total:   64,029

--- Scope del proyecto: ESI 3-5 ---
  ESI 3:  31,460  (49.1% del total con ESI)
  ESI 4:  20,182  (31.5% del total con ESI)
  ESI 5:   3,102  (4.8% del total con ESI)


---
## EDA 2 — Base rate del outcome por ESI (componentes)

In [8]:
print('=== OUTCOME BASE RATE POR ESI ===\n')
print(f'{"ESI":>5} {"N":>8} {"Outcome":>8} {"Rate%":>7} {"ADMITHOS":>9} {"TRANOTH":>8} {"DIEDED":>7}')
print('-' * 60)

rows = []
for e in range(1, 6):
    sub = df_esi[df_esi['esi'] == e]
    n       = len(sub)
    n_out   = int(sub['outcome'].sum())
    n_adm   = int((sub['ADMITHOS'] == 1).sum())
    n_tran  = int((sub['TRANOTH']  == 1).sum())
    n_died  = int((sub['DIEDED']   == 1).sum())
    rate    = n_out / n if n > 0 else 0
    print(f'{e:>5} {n:>8,} {n_out:>8,} {rate*100:>7.1f}% {n_adm:>9,} {n_tran:>8,} {n_died:>7,}')
    rows.append({'esi': e, 'n': n, 'n_outcome': n_out, 'rate': round(rate, 4),
                 'n_admithos': n_adm, 'n_tranoth': n_tran, 'n_dieded': n_died})

print('-' * 60)
n_tot   = len(df_esi)
n_out_t = int(df_esi['outcome'].sum())
print(f'{"Total":>5} {n_tot:>8,} {n_out_t:>8,} {n_out_t/n_tot*100:>7.1f}%')

# Nota sobre DIEDED
n_died_total = int((df_esi['DIEDED'] == 1).sum())
print(f'\nNota DIEDED: {n_died_total} muertes totales en ESI 1-5')
for e in [3, 4, 5]:
    sub = df_esi[df_esi['esi'] == e]
    nd  = int((sub['DIEDED'] == 1).sum())
    print(f'  ESI {e}: {nd} muertes ({nd/len(sub)*100:.2f}%)')

pd.DataFrame(rows).to_csv(REPORT_DIR / '01_outcome_rates.csv', index=False)

=== OUTCOME BASE RATE POR ESI ===

  ESI        N  Outcome   Rate%  ADMITHOS  TRANOTH  DIEDED
------------------------------------------------------------
    1      766      340    44.4%       236       47      57
    2    8,519    2,725    32.0%     2,427      291      12
    3   31,460    4,244    13.5%     3,613      637       5
    4   20,182      502     2.5%       386      115       2
    5    3,102      109     3.5%        97       11       1
------------------------------------------------------------
Total   64,029    7,920    12.4%

Nota DIEDED: 77 muertes totales en ESI 1-5
  ESI 3: 5 muertes (0.02%)
  ESI 4: 2 muertes (0.01%)
  ESI 5: 1 muertes (0.03%)


---
## EDA 3 — Missingness de variables predictoras clave

In [9]:
print('=== MISSINGNESS EN ESI 3-5 ===\n')

SET_A = ['AGE', 'SEX', 'ARREMS', 'temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT',
         'PAINSCALE', 'RFV1', 'RFV2', 'shock_index']
SET_B_EXTRA = ['ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2',
               'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB']
FAIRNESS_ONLY = ['RACERETH', 'PAYTYPER']  # excluded from all predictive models; fairness audit only (see NB00)

all_vars = SET_A + SET_B_EXTRA + FAIRNESS_ONLY

print(f'{"Variable":15s}  {"Set":32s}  {"N válido":>9}  {"% Missing":>10}')
print('-' * 75)

miss_rows = []
for col in all_vars:
    if col in SET_A:
        s = 'A'
    elif col in SET_B_EXTRA:
        s = 'B'
    else:
        s = 'fairness only - not a predictor'
    n_valid = df_main[col].notna().sum()
    pct_miss = df_main[col].isna().mean() * 100
    flag = ' <-- ALTO' if pct_miss > 30 else (' <-- REVISAR' if pct_miss > 10 else '')
    print(f'{col:15s}  {s:32s}  {n_valid:>9,}  {pct_miss:>10.1f}%{flag}')
    miss_rows.append({'variable': col, 'set': s, 'n_valid': int(n_valid), 'pct_missing': round(pct_miss, 2)})

pd.DataFrame(miss_rows).to_csv(REPORT_DIR / '01_missingness.csv', index=False)


=== MISSINGNESS EN ESI 3-5 ===

Variable         Set                                N válido   % Missing
---------------------------------------------------------------------------
AGE              A                                    54,744         0.0%
SEX              A                                    54,744         0.0%
ARREMS           A                                    54,744         0.0%
temp_f           A                                    52,935         3.3%
PULSE            A                                    52,926         3.3%
RESPR            A                                    53,231         2.8%
BPSYS            A                                    49,139        10.2% <-- REVISAR
BPDIAS           A                                    49,070        10.4% <-- REVISAR
POPCT            A                                    52,556         4.0%
PAINSCALE        A                                    41,974        23.3% <-- REVISAR
RFV1             A                         

In [10]:
# Missingness de POPCT por ESI (específicamente — es variable nueva)
print('Missingness POPCT por ESI (variable nueva vs V1):')
for e in [3, 4, 5]:
    sub  = df_main[df_main['esi'] == e]
    miss = sub['POPCT'].isna().mean() * 100
    print(f'  ESI {e}: {miss:.1f}% missing  (N={len(sub):,})')

Missingness POPCT por ESI (variable nueva vs V1):
  ESI 3: 3.2% missing  (N=31,460)
  ESI 4: 4.7% missing  (N=20,182)
  ESI 5: 7.9% missing  (N=3,102)


---
## EDA 4 — Volumen de alertas proyectado (Top 5%, 10%, 15%)

Sin modelo todavía. Esta sección calcula cuántos pacientes serían alertados  
si el modelo selecciona el top X% de cada grupo ESI.  
Permite decidir si el burden operacional es manejable antes de entrenar.

In [11]:
print('=== VOLUMEN DE ALERTAS PROYECTADO POR ESI ===\n')
print('Supuesto: se alerta el top X% por score de riesgo dentro de cada ESI')
print(f'{"ESI":>5} {"N total":>9} {"Base rate":>10} {"Top5%(N)":>10} {"Top10%(N)":>11} {"Top15%(N)":>11} {"MaxPPV@5%":>11}')
print('-' * 75)

alert_rows = []
for e in [3, 4, 5]:
    sub      = df_main[df_main['esi'] == e]
    n        = len(sub)
    n_pos    = int(sub['outcome'].sum())
    base_rate = n_pos / n

    # Alertas por año (promedio sobre 5 años de datos)
    n_per_yr  = n / len(YEARS)

    for pct in [0.05, 0.10, 0.15]:
        n_alert  = int(n * pct)
        # PPV máximo teórico: si el modelo concentra todos los positivos en el top
        max_ppv  = min(n_pos, n_alert) / n_alert if n_alert > 0 else 0
        # Enriquecimiento máximo teórico
        max_enrich = max_ppv / base_rate if base_rate > 0 else 0

    # Tabla resumen
    a5  = int(n * 0.05)
    a10 = int(n * 0.10)
    a15 = int(n * 0.15)
    # PPV máximo teórico al 5%
    max_ppv5 = min(n_pos, a5) / a5 if a5 > 0 else 0
    print(f'{e:>5} {n:>9,} {base_rate*100:>10.1f}% {a5:>8,} ({a5/n*100:.0f}%) {a10:>8,} ({a10/n*100:.0f}%) {a15:>8,} ({a15/n*100:.0f}%) {max_ppv5*100:>9.1f}%')

    for pct in [0.05, 0.10, 0.15]:
        n_alert = int(n * pct)
        ppv_max = min(n_pos, n_alert) / n_alert if n_alert > 0 else 0
        enrich  = ppv_max / base_rate if base_rate > 0 else 0
        alert_rows.append({
            'esi': e, 'n_total': n, 'n_positive': n_pos,
            'base_rate': round(base_rate, 4),
            'threshold_pct': pct,
            'n_alerted': n_alert,
            'n_alerted_per_year': round(n_alert / len(YEARS), 1),
            'max_ppv_theoretical': round(ppv_max, 4),
            'max_enrichment_theoretical': round(enrich, 2),
        })

print('\nNota: MaxPPV@5% = PPV si el modelo fuera perfecto (upper bound teórico)')
print('En la práctica el enriquecimiento real será menor — se evalúa en Notebook 02')

pd.DataFrame(alert_rows).to_csv(REPORT_DIR / '01_alert_volume_projection.csv', index=False)

=== VOLUMEN DE ALERTAS PROYECTADO POR ESI ===

Supuesto: se alerta el top X% por score de riesgo dentro de cada ESI
  ESI   N total  Base rate   Top5%(N)   Top10%(N)   Top15%(N)   MaxPPV@5%
---------------------------------------------------------------------------
    3    31,460       13.5%    1,573 (5%)    3,146 (10%)    4,719 (15%)     100.0%
    4    20,182        2.5%    1,009 (5%)    2,018 (10%)    3,027 (15%)      49.8%
    5     3,102        3.5%      155 (5%)      310 (10%)      465 (15%)      70.3%

Nota: MaxPPV@5% = PPV si el modelo fuera perfecto (upper bound teórico)
En la práctica el enriquecimiento real será menor — se evalúa en Notebook 02


In [12]:
print('=== ALERTAS PROYECTADAS POR AÑO (promedio) ===\n')
print('Ayuda a estimar carga operacional: ¿cuántas alertas por año si alertamos top X%?\n')
print(f'{"ESI":>5}  {"Base rate":>10}  {"Top 5% /año":>13}  {"Top 10% /año":>14}  {"Top 15% /año":>14}')
print('-' * 65)
for e in [3, 4, 5]:
    sub = df_main[df_main['esi'] == e]
    n   = len(sub)
    br  = sub['outcome'].mean()
    a5  = n * 0.05 / len(YEARS)
    a10 = n * 0.10 / len(YEARS)
    a15 = n * 0.15 / len(YEARS)
    print(f'{e:>5}  {br*100:>10.1f}%  {a5:>11.0f} pac  {a10:>12.0f} pac  {a15:>12.0f} pac')

=== ALERTAS PROYECTADAS POR AÑO (promedio) ===

Ayuda a estimar carga operacional: ¿cuántas alertas por año si alertamos top X%?

  ESI   Base rate    Top 5% /año    Top 10% /año    Top 15% /año
-----------------------------------------------------------------
    3        13.5%          315 pac           629 pac           944 pac
    4         2.5%          202 pac           404 pac           605 pac
    5         3.5%           31 pac            62 pac            93 pac


---
## EDA 5 — Distribución de edad por ESI + subgrupo ≥65 años

In [13]:
print('=== DISTRIBUCIÓN DE EDAD POR ESI ===\n')
print(f'{"ESI":>5} {"N":>8} {"Media":>7} {"Mediana":>8} {"p25":>6} {"p75":>6} {"p95":>6} {"≥65 N":>8} {"≥65 %":>8} {"≥65 rate_out%":>14}')
print('-' * 90)

age_rows = []
for e in [3, 4, 5]:
    sub  = df_main[df_main['esi'] == e]
    ages = sub['AGE'].dropna()
    n65  = int((sub['AGE'] >= 65).sum())
    pct65 = n65 / len(sub) * 100

    # Tasa de outcome en ≥65 vs <65
    sub65  = sub[sub['AGE'] >= 65]
    sub_lt = sub[sub['AGE'] < 65]
    rate65 = sub65['outcome'].mean() * 100 if len(sub65) > 0 else 0
    ratelt = sub_lt['outcome'].mean() * 100 if len(sub_lt) > 0 else 0

    print(f'{e:>5} {len(sub):>8,} {ages.mean():>7.1f} {ages.median():>8.1f} '
          f'{ages.quantile(0.25):>6.1f} {ages.quantile(0.75):>6.1f} {ages.quantile(0.95):>6.1f} '
          f'{n65:>8,} {pct65:>8.1f}%  {rate65:>6.1f}% (vs {ratelt:.1f}% <65)')
    age_rows.append({
        'esi': e, 'n': len(sub), 'age_mean': round(ages.mean(), 1), 'age_median': ages.median(),
        'n_65plus': n65, 'pct_65plus': round(pct65, 1),
        'outcome_rate_65plus': round(rate65, 1), 'outcome_rate_under65': round(ratelt, 1)
    })

pd.DataFrame(age_rows).to_csv(REPORT_DIR / '01_age_by_esi.csv', index=False)

=== DISTRIBUCIÓN DE EDAD POR ESI ===

  ESI        N   Media  Mediana    p25    p75    p95    ≥65 N    ≥65 %  ≥65 rate_out%
------------------------------------------------------------------------------------------
    3   31,460    41.8     40.0   23.0   60.0   83.0    6,264     19.9%    28.4% (vs 9.8% <65)
    4   20,182    30.5     28.0   11.0   47.0   72.0    1,793      8.9%     8.4% (vs 1.9% <65)
    5    3,102    29.1     26.0    7.0   46.0   73.0      304      9.8%    17.8% (vs 2.0% <65)


In [14]:
# Verificación de la hipótesis declarada en Notebook 00:
# "ESI 4 y 5, la brecha de outcome debería ser más pronunciada en ≥65"
print('=== PRE-CHECK HIPÓTESIS SUBGRUPO ≥65 (solo distribucional, sin modelo) ===\n')
print('Hipótesis: en ESI 4 y 5, los ≥65 tienen mayor tasa de outcome que los <65')
print('Si la diferencia descriptiva ya existe, el modelo debería amplificarla\n')

for e in [3, 4, 5]:
    sub   = df_main[df_main['esi'] == e]
    s65   = sub[sub['AGE'] >= 65]
    slt65 = sub[sub['AGE'] < 65]
    r65   = s65['outcome'].mean()  if len(s65)   > 0 else 0
    rlt65 = slt65['outcome'].mean() if len(slt65) > 0 else 0
    enrich = r65 / rlt65 if rlt65 > 0 else float('inf')
    print(f'ESI {e}:  ≥65 rate={r65*100:.1f}%  <65 rate={rlt65*100:.1f}%  '
          f'ratio={enrich:.2f}x  (N≥65={len(s65):,}, N<65={len(slt65):,})')

=== PRE-CHECK HIPÓTESIS SUBGRUPO ≥65 (solo distribucional, sin modelo) ===

Hipótesis: en ESI 4 y 5, los ≥65 tienen mayor tasa de outcome que los <65
Si la diferencia descriptiva ya existe, el modelo debería amplificarla

ESI 3:  ≥65 rate=28.4%  <65 rate=9.8%  ratio=2.90x  (N≥65=6,264, N<65=25,196)
ESI 4:  ≥65 rate=8.4%  <65 rate=1.9%  ratio=4.37x  (N≥65=1,793, N<65=18,389)
ESI 5:  ≥65 rate=17.8%  <65 rate=2.0%  ratio=9.04x  (N≥65=304, N<65=2,798)


In [15]:
# Histogramas de edad por ESI
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, e in zip(axes, [3, 4, 5]):
    sub  = df_main[df_main['esi'] == e]
    ages = sub['AGE'].dropna()
    ax.hist(ages, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(65, color='red', linestyle='--', linewidth=1.5, label='65 años')
    n65  = (ages >= 65).sum()
    ax.set_title(f'ESI {e}  (N={len(sub):,})\n≥65: {n65:,} ({n65/len(sub)*100:.1f}%)')
    ax.set_xlabel('Edad (años)')
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.suptitle('Distribución de edad por ESI (scope 3-5)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / '01_age_histogram.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: 01_age_histogram.png')

Guardado: 01_age_histogram.png


---
## EDA 6 — Distribución de RFV1 por ESI (top códigos)

In [16]:
# Diccionario parcial de códigos RFV de alta relevancia clínica
# (motivos de alto riesgo semántico declarados como subgrupo exploratorio en Notebook 00)
RFV_HIGH_RISK = {
    10300: 'Chest pain',
    10700: 'Shortness of breath',
    10560: 'Syncope/Fainting',
    10550: 'Weakness',
    10260: 'Abdominal pain',
    10810: 'Palpitations',
    10320: 'Chest tightness',
}

print('=== TOP 15 CÓDIGOS RFV1 POR ESI ===\n')

rfv_rows = []
for e in [3, 4, 5]:
    sub  = df_main[df_main['esi'] == e]
    n    = len(sub)
    rfv_counts = sub['RFV1'].value_counts().head(15)

    print(f'\nESI {e} (N={n:,}):')
    print(f'  {"RFV1":>7}  {"N":>7}  {"% ESI":>8}  {"Rate out%":>10}')
    print(f'  {"-"*45}')
    for code, cnt in rfv_counts.items():
        sub_rfv  = sub[sub['RFV1'] == code]
        rate_out = sub_rfv['outcome'].mean() * 100
        hr_label = f'  ** {RFV_HIGH_RISK[code]}' if code in RFV_HIGH_RISK else ''
        print(f'  {int(code):>7}  {cnt:>7,}  {cnt/n*100:>8.1f}%  {rate_out:>9.1f}%{hr_label}')
        rfv_rows.append({'esi': e, 'rfv1': int(code), 'n': cnt, 'pct': round(cnt/n*100, 2), 'outcome_rate': round(rate_out, 2)})

pd.DataFrame(rfv_rows).to_csv(REPORT_DIR / '01_rfv1_by_esi.csv', index=False)

=== TOP 15 CÓDIGOS RFV1 POR ESI ===


ESI 3 (N=31,460):
     RFV1        N     % ESI   Rate out%
  ---------------------------------------------
    15451    3,364      10.7%       15.1%
    10501    1,581       5.0%       15.6%
    10100    1,091       3.5%       10.2%
    12100    1,025       3.3%        6.0%
    14150      986       3.1%       29.3%
    14400      921       2.9%        9.2%
    15300      777       2.5%       12.7%
    12250      683       2.2%       13.0%
    15250      674       2.1%       14.5%
    10552      669       2.1%        8.5%
    19051      620       2.0%       11.3%
    15452      493       1.6%       11.8%
    55050      477       1.5%        6.3%
    19201      452       1.4%       12.6%
    58100      426       1.4%       15.5%

ESI 4 (N=20,182):
     RFV1        N     % ESI   Rate out%
  ---------------------------------------------
    10100    1,147       5.7%        2.2%
    14400    1,100       5.5%        1.1%
    19051      632       3.1%    

In [17]:
# Foco en motivos de alto riesgo semántico con ESI bajo (subgrupo exploratorio)
print('=== MOTIVOS DE ALTO RIESGO SEMÁNTICO EN ESI 4-5 ===\n')
print('Subgrupo exploratorio del Notebook 00: ¿qué tasa de outcome tienen estos motivos en ESI bajo?\n')

for e in [4, 5]:
    sub  = df_main[df_main['esi'] == e]
    br   = sub['outcome'].mean()
    print(f'ESI {e} (base rate {br*100:.1f}%):')
    for code, label in RFV_HIGH_RISK.items():
        s = sub[sub['RFV1'] == code]
        if len(s) == 0:
            continue
        rate  = s['outcome'].mean()
        enr   = rate / br if br > 0 else 0
        print(f'  {label:<30}  N={len(s):>5,}  rate={rate*100:.1f}%  enrich={enr:.2f}x')
    print()

=== MOTIVOS DE ALTO RIESGO SEMÁNTICO EN ESI 4-5 ===

Subgrupo exploratorio del Notebook 00: ¿qué tasa de outcome tienen estos motivos en ESI bajo?

ESI 4 (base rate 2.5%):
  Chest pain                      N=   30  rate=6.7%  enrich=2.68x
  Shortness of breath             N=    5  rate=0.0%  enrich=0.00x
  Weakness                        N=   19  rate=0.0%  enrich=0.00x

ESI 5 (base rate 3.5%):
  Chest pain                      N=    9  rate=11.1%  enrich=3.16x
  Shortness of breath             N=    5  rate=0.0%  enrich=0.00x
  Weakness                        N=    3  rate=0.0%  enrich=0.00x



---
## EDA 7 — Distribución de comorbilidades por ESI

In [18]:
COMORBID_COLS = ['ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1',
                 'DIABTYP2', 'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB']

print('=== PREVALENCIA DE COMORBILIDADES POR ESI (%) ===\n')
print(f'{"Comorbilidad":15s}', end='')
for e in [3, 4, 5]: print(f'  ESI{e}(%)', end='')
print('  Global(%)')
print('-' * 60)

comorbid_rows = []
for col in COMORBID_COLS:
    print(f'{col:15s}', end='')
    row = {'comorbidity': col}
    for e in [3, 4, 5]:
        sub = df_main[df_main['esi'] == e]
        pct = (sub[col] == 1).mean() * 100
        row[f'esi{e}_pct'] = round(pct, 1)
        print(f'  {pct:>8.1f}', end='')
    global_pct = (df_main[col] == 1).mean() * 100
    row['global_pct'] = round(global_pct, 1)
    print(f'  {global_pct:>8.1f}')
    comorbid_rows.append(row)

pd.DataFrame(comorbid_rows).to_csv(REPORT_DIR / '01_comorbidities_by_esi.csv', index=False)

=== PREVALENCIA DE COMORBILIDADES POR ESI (%) ===

Comorbilidad     ESI3(%)  ESI4(%)  ESI5(%)  Global(%)
------------------------------------------------------------
ASTHMA               10.0       9.0       8.2       9.5
CANCER                4.6       1.6       1.4       3.3
COPD                  6.4       3.1       3.0       5.0
CHF                   4.1       1.2       1.6       2.9
DEPRN                11.7       8.1       6.5      10.1
DIABTYP1              0.7       0.4       0.3       0.6
DIABTYP2              7.4       3.5       3.4       5.7
ESRD                  1.1       0.3       0.5       0.7
EDHIV                 0.8       0.5       0.6       0.7
HTN                  28.8      15.9      15.5      23.3
OBESITY               6.2       3.8       3.2       5.1
SUBSTAB               7.0       4.7       4.8       6.0


In [19]:
# Distribución del score de carga de comorbilidades por ESI
df_main = df_main.copy()
df_main['comorbidity_score'] = df_main[COMORBID_COLS].apply(
    lambda row: (row == 1).sum(), axis=1
)

print('=== SCORE DE CARGA DE COMORBILIDADES POR ESI ===\n')
print('(Subgrupo exploratorio del Notebook 00: carga crónica independiente de edad)\n')
print(f'{"ESI":>5} {"Score=0":>9} {"Score=1":>9} {"Score≥2":>9} {"Media":>7} {"Rate@S0":>9} {"Rate@S1":>9} {"Rate@S≥2":>10}')
print('-' * 75)

for e in [3, 4, 5]:
    sub = df_main[df_main['esi'] == e]
    s0  = sub[sub['comorbidity_score'] == 0]
    s1  = sub[sub['comorbidity_score'] == 1]
    s2p = sub[sub['comorbidity_score'] >= 2]
    mean_s = sub['comorbidity_score'].mean()
    r0  = s0['outcome'].mean()  * 100 if len(s0)  > 0 else 0
    r1  = s1['outcome'].mean()  * 100 if len(s1)  > 0 else 0
    r2p = s2p['outcome'].mean() * 100 if len(s2p) > 0 else 0
    print(f'{e:>5} {len(s0):>7,} ({len(s0)/len(sub)*100:.0f}%) '
          f'{len(s1):>7,} ({len(s1)/len(sub)*100:.0f}%) '
          f'{len(s2p):>7,} ({len(s2p)/len(sub)*100:.0f}%)  '
          f'{mean_s:>5.1f}   {r0:>7.1f}%  {r1:>7.1f}%  {r2p:>8.1f}%')

=== SCORE DE CARGA DE COMORBILIDADES POR ESI ===

(Subgrupo exploratorio del Notebook 00: carga crónica independiente de edad)

  ESI   Score=0   Score=1   Score≥2   Media   Rate@S0   Rate@S1   Rate@S≥2
---------------------------------------------------------------------------
    3  15,493 (49%)   8,593 (27%)   7,374 (23%)    0.9       7.5%     14.3%      25.1%
    4  13,387 (66%)   4,289 (21%)   2,506 (12%)    0.5       1.5%      3.2%       6.7%
    5   2,144 (69%)     602 (19%)     356 (11%)    0.5       1.3%      5.5%      13.8%


---
## EDA 8 — Distribución demográfica (RACERETH, SEX, PAYTYPER) por ESI

In [20]:
# Diccionarios de etiquetas (según doc CDC)
RACERETH_LABELS = {
    1: 'Hispanic',
    2: 'Non-Hisp White',
    3: 'Non-Hisp Black',
    4: 'Non-Hisp Other',
}

SEX_LABELS = {1: 'Male', 2: 'Female'}

PAYTYPE_LABELS = {
    1: 'Private insurance',
    2: 'Medicare',
    3: 'Medicaid/CHIP',
    4: 'Worker comp',
    5: 'Self-pay',
    6: 'No charge',
    7: 'Other',
    8: 'Unknown',
    -9: 'Blank',
}

def print_demo_by_esi(col, labels, title):
    print(f'\n=== {title} ===\n')
    rows = []
    for e in [3, 4, 5]:
        sub = df_main[df_main['esi'] == e]
        n   = len(sub)
        print(f'ESI {e} (N={n:,}):')
        for code, label in labels.items():
            sub_g  = sub[sub[col] == code]
            ng     = len(sub_g)
            rate_g = sub_g['outcome'].mean() * 100 if ng > 0 else 0
            global_rate = sub['outcome'].mean() * 100
            enr = rate_g / global_rate if global_rate > 0 else 0
            print(f'  {label:<22}: {ng:>6,} ({ng/n*100:>5.1f}%)  outcome={rate_g:.1f}%  enrich={enr:.2f}x')
            rows.append({'esi': e, 'variable': col, 'code': code, 'label': label,
                         'n': ng, 'pct': round(ng/n*100, 1), 'outcome_rate': round(rate_g, 2)})
        print()
    return rows

all_rows = []
all_rows += print_demo_by_esi('RACERETH', RACERETH_LABELS, 'RAZA/ETNIA POR ESI')
all_rows += print_demo_by_esi('SEX',      SEX_LABELS,      'SEXO POR ESI')

pd.DataFrame(all_rows).to_csv(REPORT_DIR / '01_demographics_by_esi.csv', index=False)


=== RAZA/ETNIA POR ESI ===

ESI 3 (N=31,460):
  Hispanic              : 17,961 ( 57.1%)  outcome=15.6%  enrich=1.16x
  Non-Hisp White        :  7,313 ( 23.2%)  outcome=9.8%  enrich=0.72x
  Non-Hisp Black        :  4,859 ( 15.4%)  outcome=11.2%  enrich=0.83x
  Non-Hisp Other        :  1,327 (  4.2%)  outcome=13.9%  enrich=1.03x

ESI 4 (N=20,182):
  Hispanic              : 10,609 ( 52.6%)  outcome=2.9%  enrich=1.18x
  Non-Hisp White        :  5,198 ( 25.8%)  outcome=1.8%  enrich=0.74x
  Non-Hisp Black        :  3,645 ( 18.1%)  outcome=2.3%  enrich=0.94x
  Non-Hisp Other        :    730 (  3.6%)  outcome=1.4%  enrich=0.55x

ESI 5 (N=3,102):
  Hispanic              :  1,545 ( 49.8%)  outcome=3.8%  enrich=1.07x
  Non-Hisp White        :    868 ( 28.0%)  outcome=3.8%  enrich=1.08x
  Non-Hisp Black        :    590 ( 19.0%)  outcome=2.4%  enrich=0.68x
  Non-Hisp Other        :     99 (  3.2%)  outcome=4.0%  enrich=1.15x


=== SEXO POR ESI ===

ESI 3 (N=31,460):
  Male                  : 17,94

In [21]:
# Tipo de seguro por ESI
print('=== TIPO DE SEGURO (PAYTYPER) POR ESI ===\n')
pay_rows = []
for e in [3, 4, 5]:
    sub = df_main[df_main['esi'] == e]
    n   = len(sub)
    print(f'ESI {e} (N={n:,}):')
    vc = sub['PAYTYPER'].value_counts().sort_index()
    for code, cnt in vc.items():
        label = PAYTYPE_LABELS.get(int(code), f'code_{int(code)}')
        sub_p = sub[sub['PAYTYPER'] == code]
        rate  = sub_p['outcome'].mean() * 100 if len(sub_p) > 0 else 0
        print(f'  {label:<22}: {cnt:>6,} ({cnt/n*100:>5.1f}%)  outcome={rate:.1f}%')
        pay_rows.append({'esi': e, 'paytyper': int(code), 'label': label, 'n': cnt, 'outcome_rate': round(rate, 2)})
    print()

pd.DataFrame(pay_rows).to_csv(REPORT_DIR / '01_paytype_by_esi.csv', index=False)

=== TIPO DE SEGURO (PAYTYPER) POR ESI ===

ESI 3 (N=31,460):
  Blank                 :    352 (  1.1%)  outcome=16.2%
  code_-8               :  2,317 (  7.4%)  outcome=10.5%
  Private insurance     :  8,315 ( 26.4%)  outcome=10.5%
  Medicare              :  6,897 ( 21.9%)  outcome=26.2%
  Medicaid/CHIP         : 10,392 ( 33.0%)  outcome=9.5%
  Worker comp           :    148 (  0.5%)  outcome=8.1%
  Self-pay              :  2,261 (  7.2%)  outcome=7.3%
  No charge             :    135 (  0.4%)  outcome=5.9%
  Other                 :    643 (  2.0%)  outcome=12.9%

ESI 4 (N=20,182):
  Blank                 :    275 (  1.4%)  outcome=3.3%
  code_-8               :  1,576 (  7.8%)  outcome=2.2%
  Private insurance     :  4,977 ( 24.7%)  outcome=2.3%
  Medicare              :  2,272 ( 11.3%)  outcome=6.5%
  Medicaid/CHIP         :  8,571 ( 42.5%)  outcome=1.9%
  Worker comp           :    239 (  1.2%)  outcome=0.4%
  Self-pay              :  1,788 (  8.9%)  outcome=1.1%
  No charge        

---
## EDA 9 — Distribución de vitales por ESI

In [22]:
VITALS = [
    ('temp_f',      'Temperatura (°F)',    95.0, 104.0),
    ('PULSE',       'Pulso (lpm)',          40,   200),
    ('RESPR',       'Frec. resp (/min)',     8,    40),
    ('BPSYS',       'PAS (mmHg)',           60,   250),
    ('BPDIAS',      'PAD (mmHg)',           30,   150),
    ('POPCT',       'SpO2 (%)',             80,   100),
    ('PAINSCALE',   'Dolor (0-10)',          0,    10),
    ('shock_index', 'Shock index',          0.2,   2.5),
]

print('=== VITALES POR ESI (mediana [p25-p75]) ===\n')
print(f'{"Variable":15s}  {"ESI 3":>16}  {"ESI 4":>16}  {"ESI 5":>16}')
print('-' * 70)

vital_rows = []
for col, label, lo, hi in VITALS:
    print(f'{label[:15]:15s}', end='  ')
    for e in [3, 4, 5]:
        sub = df_main[df_main['esi'] == e]
        s   = sub[col].dropna()
        s   = s[s.between(lo, hi)]
        if len(s) == 0:
            print(f'{"N/A":>16}', end='  ')
        else:
            med = s.median()
            p25 = s.quantile(0.25)
            p75 = s.quantile(0.75)
            cell_str = f'{med:.1f} [{p25:.1f}-{p75:.1f}]'
            print(f'{cell_str:>16}', end='  ')
            vital_rows.append({'variable': col, 'esi': e, 'median': med, 'p25': p25, 'p75': p75,
                               'n_valid': len(s), 'pct_missing': round(sub[col].isna().mean()*100, 1)})
    print()

pd.DataFrame(vital_rows).to_csv(REPORT_DIR / '01_vitals_by_esi.csv', index=False)

=== VITALES POR ESI (mediana [p25-p75]) ===

Variable                    ESI 3             ESI 4             ESI 5
----------------------------------------------------------------------
Temperatura (°F  98.2 [97.8-98.6]  98.2 [97.9-98.6]  98.2 [97.8-98.6]  
Pulso (lpm)      87.0 [76.0-101.0]  89.0 [77.0-104.0]  88.0 [77.0-104.0]  
Frec. resp (/mi  18.0 [16.0-20.0]  18.0 [16.0-20.0]  18.0 [16.0-20.0]  
PAS (mmHg)       133.0 [119.0-149.0]  129.0 [116.0-144.0]  128.0 [114.0-143.0]  
PAD (mmHg)       79.0 [69.0-88.0]  77.0 [69.0-86.0]  76.0 [67.0-86.0]  
SpO2 (%)         

98.0 [97.0-99.0]  98.0 [97.0-100.0]  98.0 [97.0-100.0]  
Dolor (0-10)        5.0 [0.0-8.0]     5.0 [0.0-8.0]     2.0 [0.0-7.0]  
Shock index         0.6 [0.5-0.8]     0.7 [0.6-0.8]     0.7 [0.6-0.8]  


---
## EDA 10 — Resumen final y decisión de threshold

In [23]:
print('=' * 65)
print('RESUMEN EDA — NÚMEROS CLAVE PARA DISEÑO DE ALERTAS')
print('=' * 65)

print('\n1. VOLUMEN Y BASE RATE')
for e in [3, 4, 5]:
    sub = df_main[df_main['esi'] == e]
    br  = sub['outcome'].mean()
    print(f'   ESI {e}: N={len(sub):,}  base rate={br*100:.1f}%  (≥65: {(sub["AGE"]>=65).sum():,} pac)')

print('\n2. UMBRAL DE ALERTA (principio: enriquecimiento ≥1.5x, burden manejable)')
for e in [3, 4, 5]:
    sub  = df_main[df_main['esi'] == e]
    br   = sub['outcome'].mean()
    n    = len(sub)
    # Cuántos % necesita el modelo para capturar todos los positivos con 1.5x
    # target_ppv = 1.5 * br
    # Con un modelo perfecto, top X% captura todos si X% >= base_rate
    # Operacionalmente: top 10-15% es manejable; ≤5% puede ser muy restrictivo
    target_ppv = 1.5 * br
    print(f'   ESI {e}: target PPV para 1.5x = {target_ppv*100:.1f}%  '
          f'(top 10% = {n*0.10/len(YEARS):.0f} pac/año en los datos)')

print('\n3. SUBGRUPO ≥65 — tasas crudas de outcome por grupo etario (descriptivo)')
for e in [4, 5]:
    sub   = df_main[df_main['esi'] == e]
    s65   = sub[sub['AGE'] >= 65]
    slt65 = sub[sub['AGE'] < 65]
    r65   = s65['outcome'].mean()  if len(s65)   > 0 else 0
    rlt65 = slt65['outcome'].mean() if len(slt65) > 0 else 0
    enr   = r65 / rlt65 if rlt65 > 0 else 0
    print(f'   ESI {e}: rate≥65={r65*100:.1f}%  rate<65={rlt65*100:.1f}%  ratio={enr:.2f}x')

print('   Nota: tasas crudas de outcome por grupo etario (descriptivo). La prueba formal')
print('   de la hipótesis ≥65 usa el enriquecimiento del modelo y se reporta en NB05;')
print('   el resultado allí matiza la hipótesis original (ver NB05, sección de subgrupo etario).')

print('\n4. NOTA: el threshold operacional se define en Notebook 03 con los resultados del modelo')
print('   El EDA da el contexto; el modelo da el score real para fijar el corte')

RESUMEN EDA — NÚMEROS CLAVE PARA DISEÑO DE ALERTAS

1. VOLUMEN Y BASE RATE
   ESI 3: N=31,460  base rate=13.5%  (≥65: 6,264 pac)
   ESI 4: N=20,182  base rate=2.5%  (≥65: 1,793 pac)
   ESI 5: N=3,102  base rate=3.5%  (≥65: 304 pac)

2. UMBRAL DE ALERTA (principio: enriquecimiento ≥1.5x, burden manejable)
   ESI 3: target PPV para 1.5x = 20.2%  (top 10% = 629 pac/año en los datos)
   ESI 4: target PPV para 1.5x = 3.7%  (top 10% = 404 pac/año en los datos)
   ESI 5: target PPV para 1.5x = 5.3%  (top 10% = 62 pac/año en los datos)

3. SUBGRUPO ≥65 — tasas crudas de outcome por grupo etario (descriptivo)
   ESI 4: rate≥65=8.4%  rate<65=1.9%  ratio=4.37x
   ESI 5: rate≥65=17.8%  rate<65=2.0%  ratio=9.04x
   Nota: tasas crudas de outcome por grupo etario (descriptivo). La prueba formal
   de la hipótesis ≥65 usa el enriquecimiento del modelo y se reporta en NB05;
   el resultado allí matiza la hipótesis original (ver NB05, sección de subgrupo etario).

4. NOTA: el threshold operacional se de

In [24]:
# Guardar resumen JSON para referencia de notebooks siguientes
summary = {
    'notebook': '01_eda_nhamcs',
    'years': YEARS,
    'n_total_esi35': int(len(df_main)),
    'by_esi': {}
}

for e in [3, 4, 5]:
    sub = df_main[df_main['esi'] == e]
    summary['by_esi'][str(e)] = {
        'n': int(len(sub)),
        'n_positive': int(sub['outcome'].sum()),
        'base_rate': round(float(sub['outcome'].mean()), 4),
        'n_65plus': int((sub['AGE'] >= 65).sum()),
        'pct_65plus': round(float((sub['AGE'] >= 65).mean() * 100), 1),
        'outcome_rate_65plus': round(float(sub[sub['AGE']>=65]['outcome'].mean() if (sub['AGE']>=65).sum()>0 else 0), 4),
        'outcome_rate_under65': round(float(sub[sub['AGE']<65]['outcome'].mean() if (sub['AGE']<65).sum()>0 else 0), 4),
        'n_alerted_top10pct': int(len(sub) * 0.10),
        'target_ppv_15x_enrichment': round(float(sub['outcome'].mean() * 1.5), 4),
    }

out_json = REPORT_DIR / '01_eda_summary.json'
with open(out_json, 'w') as f:
    json.dump(summary, f, indent=2)

print('Resumen EDA guardado:')
print(json.dumps(summary, indent=2))

Resumen EDA guardado:
{
  "notebook": "01_eda_nhamcs",
  "years": [
    2016,
    2017,
    2018,
    2019,
    2022
  ],
  "n_total_esi35": 54744,
  "by_esi": {
    "3": {
      "n": 31460,
      "n_positive": 4244,
      "base_rate": 0.1349,
      "n_65plus": 6264,
      "pct_65plus": 19.9,
      "outcome_rate_65plus": 0.2837,
      "outcome_rate_under65": 0.0979,
      "n_alerted_top10pct": 3146,
      "target_ppv_15x_enrichment": 0.2024
    },
    "4": {
      "n": 20182,
      "n_positive": 502,
      "base_rate": 0.0249,
      "n_65plus": 1793,
      "pct_65plus": 8.9,
      "outcome_rate_65plus": 0.0837,
      "outcome_rate_under65": 0.0191,
      "n_alerted_top10pct": 2018,
      "target_ppv_15x_enrichment": 0.0373
    },
    "5": {
      "n": 3102,
      "n_positive": 109,
      "base_rate": 0.0351,
      "n_65plus": 304,
      "pct_65plus": 9.8,
      "outcome_rate_65plus": 0.1776,
      "outcome_rate_under65": 0.0197,
      "n_alerted_top10pct": 310,
      "target_ppv_15x_en

---
## Archivos generados

| Archivo | Descripción |
|---------|-------------|
| `data/processed/nhamcs_pooled_2016_2022.csv` | Dataset procesado ESI 3-5, listo para Notebook 02 |
| `reports/01_eda_nhamcs/01_esi_by_year.csv` | N por ESI y año |
| `reports/01_eda_nhamcs/01_outcome_rates.csv` | Base rates y componentes del outcome |
| `reports/01_eda_nhamcs/01_missingness.csv` | Missingness de predictores |
| `reports/01_eda_nhamcs/01_alert_volume_projection.csv` | Volumen de alertas Top 5/10/15% |
| `reports/01_eda_nhamcs/01_age_by_esi.csv` | Distribución de edad y subgrupo ≥65 |
| `reports/01_eda_nhamcs/01_age_histogram.png` | Histogramas de edad por ESI |
| `reports/01_eda_nhamcs/01_rfv1_by_esi.csv` | Top RFV1 por ESI con tasa de outcome |
| `reports/01_eda_nhamcs/01_comorbidities_by_esi.csv` | Prevalencia de comorbilidades |
| `reports/01_eda_nhamcs/01_demographics_by_esi.csv` | Distribución demográfica |
| `reports/01_eda_nhamcs/01_paytype_by_esi.csv` | Tipo de seguro por ESI |
| `reports/01_eda_nhamcs/01_vitals_by_esi.csv` | Distribución de vitales |
| `reports/01_eda_nhamcs/01_eda_summary.json` | Resumen JSON para notebooks siguientes |

**Próximo paso:** Notebook 02 — entrenamiento LightGBM intra-ESI, tres sets de features, validación.

---
## EDA 11 — Auditoría de valores especiales y missingness real

NHAMCS codifica faltantes con: **-9 = Blank, -8 = Unknown**.  
RFV2 usa -9 cuando no hay segunda razón de visita (N muy alto es normal).  
Esta sección verifica que la limpieza los convirtió correctamente a NaN,  
y aplica los filtros de Harvard (Raita 2019) para vitales fisiológicamente imposibles.

In [25]:
# Verificación post-limpieza: ¿quedaron valores especiales como números?
# Nota: 98/99 son valores CLÍNICOS VÁLIDOS para PULSE, BPSYS, BPDIAS, POPCT.
# Los únicos códigos especiales reales en NHAMCS son -9 y -8.

print('=== EDA 11 — VERIFICACIÓN POST-LIMPIEZA (valores especiales) ===\n')

VITAL_COLS  = ['PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'PAINSCALE', 'temp_f']
MISSING_CODES = [-9, -8, -7]

print('Vitales — códigos de missing NHAMCS (-9/-8/-7):')
for col in VITAL_COLS:
    if col not in df_main.columns:
        continue
    issues = []
    for code in MISSING_CODES:
        n = int((df_main[col] == code).sum())
        if n > 0:
            issues.append(f'{code}->{n}')
    status = 'LIMPIO' if not issues else 'PROBLEMA: ' + str(issues)
    print(f'  {col:12s}: {status}')

# RFV: -9 = no hay razón de visita registrada
print()
print('RFV — código -9 (sin razón registrada):')
for col in ['RFV1', 'RFV2']:
    n_neg = int((df_main[col] < 0).sum())
    n_nan = int(df_main[col].isna().sum())
    print(f'  {col}: valores negativos={n_neg:,}  NaN={n_nan:,}')

# Acción correctiva: convertir RFV negativos a NaN
df_main = df_main.copy()
for col in ['RFV1', 'RFV2']:
    n_fix = int((df_main[col] < 0).sum())
    df_main[col] = df_main[col].where(df_main[col] > 0)
    if n_fix > 0:
        print(f'  {col}: {n_fix:,} valores negativos -> NaN (acción correctiva)')

# Comorbilidades: 1=Yes, 2=No, -9/-8=Unknown
# Binarizar: 1->1, 2->0, el resto->NaN
print()
print('Comorbilidades — binarización 1=Yes / 0=No / NaN=desconocido:')
for col in COMORBID_COLS:
    df_main[col] = df_main[col].map({1: 1, 2: 0, 0: 0}).astype('Int8')
    n1 = int((df_main[col] == 1).sum())
    n0 = int((df_main[col] == 0).sum())
    nm = int(df_main[col].isna().sum())
    print(f'  {col:12s}: Yes={n1:,}  No={n0:,}  NaN={nm}')


=== EDA 11 — VERIFICACIÓN POST-LIMPIEZA (valores especiales) ===

Vitales — códigos de missing NHAMCS (-9/-8/-7):
  PULSE       : LIMPIO
  RESPR       : LIMPIO
  BPSYS       : LIMPIO
  BPDIAS      : LIMPIO
  POPCT       : LIMPIO
  PAINSCALE   : LIMPIO
  temp_f      : LIMPIO

RFV — código -9 (sin razón registrada):
  RFV1: valores negativos=25  NaN=0
  RFV2: valores negativos=17,241  NaN=0
  RFV1: 25 valores negativos -> NaN (acción correctiva)
  RFV2: 17,241 valores negativos -> NaN (acción correctiva)

Comorbilidades — binarización 1=Yes / 0=No / NaN=desconocido:
  ASTHMA      : Yes=5,212  No=49,532  NaN=0
  CANCER      : Yes=1,819  No=52,925  NaN=0


  COPD        : Yes=2,723  No=52,021  NaN=0
  CHF         : Yes=1,561  No=53,183  NaN=0
  DEPRN       : Yes=5,503  No=49,241  NaN=0
  DIABTYP1    : Yes=302  No=54,442  NaN=0
  DIABTYP2    : Yes=3,145  No=51,599  NaN=0
  ESRD        : Yes=404  No=54,340  NaN=0
  EDHIV       : Yes=365  No=54,379  NaN=0
  HTN         : Yes=12,750  No=41,994  NaN=0
  OBESITY     : Yes=2,805  No=51,939  NaN=0
  SUBSTAB     : Yes=3,284  No=51,460  NaN=0


In [26]:
# Filtros de Harvard (Raita 2019 Critical Care) — valores fisiológicamente imposibles
# Se convierten a NaN para que LightGBM los maneje nativamente (no se eliminan filas)

HARVARD_FILTERS = {
    'BPSYS':  ('>', 300),
    'BPDIAS': ('>', 200),
    'PULSE':  ('>', 300),
    'RESPR':  ('>', 80),
    'POPCT':  ('>', 100),
}

print('=== FILTROS HARVARD (Raita 2019) ===\n')

n_total = len(df_main)
any_mask = pd.Series(False, index=df_main.index)

for col, (op, thr) in HARVARD_FILTERS.items():
    if col not in df_main.columns:
        continue
    mask = df_main[col] > thr
    n    = int(mask.sum())
    pct  = n / n_total * 100
    print(f'  {col:12s} > {thr:>3}: {n:>5,} valores ({pct:.3f}%) -> NaN')
    df_main.loc[mask, col] = np.nan
    any_mask = any_mask | mask

n_affected = int(any_mask.sum())
print(f'\n  Filas con al menos 1 vital fuera de rango Harvard: {n_affected:,} ({n_affected/n_total*100:.2f}%)')
print(f'  Estrategia: NaN en lugar de drop — LightGBM maneja missing nativo')
print(f'  N total ESI 3-5 sin cambio: {len(df_main):,}')


=== FILTROS HARVARD (Raita 2019) ===

  BPSYS        > 300:     0 valores (0.000%) -> NaN
  BPDIAS       > 200:     0 valores (0.000%) -> NaN
  PULSE        > 300:     0 valores (0.000%) -> NaN
  RESPR        >  80:    64 valores (0.117%) -> NaN
  POPCT        > 100:     0 valores (0.000%) -> NaN

  Filas con al menos 1 vital fuera de rango Harvard: 64 (0.12%)
  Estrategia: NaN en lugar de drop — LightGBM maneja missing nativo
  N total ESI 3-5 sin cambio: 54,744


---
## EDA 12 — Auditoría de rangos clínicos

Para cada vital: mínimo, p1, p5, mediana, p95, p99, máximo.  
Identifica valores fuera de rango clínicamente posible post-filtrado Harvard.

In [27]:
CLINICAL_BOUNDS = {
    'temp_f':     (94.0, 106.0),
    'PULSE':      (20,   300),
    'RESPR':      (4,    80),
    'BPSYS':      (40,   300),
    'BPDIAS':     (10,   200),
    'POPCT':      (0,    100),
    'PAINSCALE':  (0,    10),
    'shock_index':(0.05, 5.0),
}

print('=== AUDITORÍA DE RANGOS CLÍNICOS (ESI 3-5, post-limpieza Harvard) ===\n')
print(f'{"Variable":12s}  {"N":>8}  {"Miss%":>6}  {"Min":>6}  {"p1":>6}  '
      f'{"p5":>6}  {"Med":>6}  {"p95":>6}  {"p99":>6}  {"Max":>8}  Estado')
print('-' * 110)

range_rows = []
for col, (lo, hi) in CLINICAL_BOUNDS.items():
    s = df_main[col].dropna()
    if len(s) == 0:
        print(f'{col:12s}: SIN DATOS')
        continue
    pm  = df_main[col].isna().mean() * 100
    mn  = s.min()
    p1  = s.quantile(0.01)
    p5  = s.quantile(0.05)
    med = s.median()
    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)
    mx  = s.max()
    n_ext = int(((s > hi) | (s < lo)).sum())
    flag  = f'<-- {n_ext} outliers' if n_ext > 0 else 'OK'
    print(f'{col:12s}  {len(s):>8,}  {pm:>6.1f}%  {mn:>6.1f}  {p1:>6.1f}  '
          f'{p5:>6.1f}  {med:>6.1f}  {p95:>6.1f}  {p99:>6.1f}  {mx:>8.1f}  {flag}')
    range_rows.append({'variable': col, 'n_valid': len(s), 'pct_missing': round(pm, 1),
                       'min': mn, 'p1': p1, 'p5': p5, 'median': med,
                       'p95': p95, 'p99': p99, 'max': mx, 'n_outside_bounds': n_ext})

pd.DataFrame(range_rows).to_csv(REPORT_DIR / '01_clinical_range_audit.csv', index=False)
print('\nGuardado: 01_clinical_range_audit.csv')


=== AUDITORÍA DE RANGOS CLÍNICOS (ESI 3-5, post-limpieza Harvard) ===

Variable             N   Miss%     Min      p1      p5     Med     p95     p99       Max  Estado
--------------------------------------------------------------------------------------------------------------
temp_f          52,935     3.3%    90.0    96.5    97.1    98.2   100.0   102.7     109.0  <-- 19 outliers
PULSE           52,926     3.3%     1.0    54.0    62.0    88.0   136.0   168.0     228.0  <-- 48 outliers
RESPR           53,167     2.9%     1.0    14.0    16.0    18.0    28.0    42.0      80.0  <-- 14 outliers
BPSYS           49,139    10.2%    64.0    90.0   101.0   131.0   177.0   200.0     287.0  OK
BPDIAS          49,070    10.4%    22.0    48.0    57.0    78.0   102.0   116.0     190.0  OK
POPCT           52,556     4.0%     0.0    88.0    94.0    98.0   100.0   100.0     100.0  OK


PAINSCALE       41,974    23.3%     0.0     0.0     0.0     5.0    10.0    10.0      10.0  OK
shock_index     47,919    12.5%     0.1     0.3     0.4     0.7     1.1     1.4       2.7  OK

Guardado: 01_clinical_range_audit.csv


---
## EDA 13 — Features derivadas

Tres features derivadas para el modelo:  
- `shock_index` = PULSE / BPSYS — marcador de inestabilidad hemodinámica  
- `comorbidity_count` = suma de comorbilidades presentes — carga crónica total  
- `age_65plus` = 1 si AGE ≥ 65, 0 si no — variable del subgrupo primario declarado en NB 00

In [28]:
# Reconstruir features derivadas post-limpieza Harvard
df_main = df_main.copy()

# shock_index post-Harvard (BPSYS ya filtrado)
df_main['shock_index'] = np.where(
    (df_main['BPSYS'].isna()) | (df_main['BPSYS'] == 0),
    np.nan,
    df_main['PULSE'] / df_main['BPSYS']
)
df_main['shock_index'] = df_main['shock_index'].where(df_main['shock_index'].between(0.1, 5.0))

# comorbidity_count: suma de condiciones presentes
df_main['comorbidity_count'] = df_main[COMORBID_COLS].apply(
    lambda row: int((row == 1).sum()), axis=1
)

# age_65plus
df_main['age_65plus'] = (df_main['AGE'] >= 65).astype('Int8')

print('=== DISTRIBUCIÓN DE FEATURES DERIVADAS POR ESI ===\n')
print('--- shock_index ---')
print(f'{"ESI":>5}  {"N válido":>9}  {"Miss%":>6}  {"p5":>6}  {"Mediana":>8}  {"p95":>7}  {"Max":>7}')
for e in [3, 4, 5]:
    sub  = df_main[df_main['esi'] == e]
    s    = sub['shock_index'].dropna()
    miss = sub['shock_index'].isna().mean() * 100
    print(f'{e:>5}  {len(s):>9,}  {miss:>6.1f}%  {s.quantile(0.05):>6.3f}  '
          f'{s.median():>8.3f}  {s.quantile(0.95):>7.3f}  {s.max():>7.3f}')

print('\n--- comorbidity_count ---')
print(f'{"ESI":>5}  {"Media":>7}  {"Score=0 (%)": >14}  {"Score=1 (%)": >14}  {"Score>=2 (%)": >15}  {"Max":>5}')
for e in [3, 4, 5]:
    sub = df_main[df_main['esi'] == e]['comorbidity_count']
    s0  = (sub == 0).sum()
    s1  = (sub == 1).sum()
    s2p = (sub >= 2).sum()
    print(f'{e:>5}  {sub.mean():>7.2f}  {s0:>7,} ({s0/len(sub)*100:.0f}%)'
          f'  {s1:>7,} ({s1/len(sub)*100:.0f}%)'
          f'  {s2p:>8,} ({s2p/len(sub)*100:.0f}%)'
          f'  {int(sub.max()):>5}')

print('\n--- age_65plus ---')
print(f'{"ESI":>5}  {"N >=65":>8}  {"% >=65":>8}  {"Rate outcome >=65":>18}  {"Rate outcome <65":>18}')
for e in [3, 4, 5]:
    sub  = df_main[df_main['esi'] == e]
    s65  = sub[sub['age_65plus'] == 1]
    slt  = sub[sub['age_65plus'] == 0]
    pct  = len(s65) / len(sub) * 100
    r65  = s65['outcome'].mean() * 100 if len(s65) > 0 else 0
    rlt  = slt['outcome'].mean()  * 100 if len(slt)  > 0 else 0
    print(f'{e:>5}  {len(s65):>8,}  {pct:>8.1f}%  {r65:>16.1f}%  {rlt:>16.1f}%')


=== DISTRIBUCIÓN DE FEATURES DERIVADAS POR ESI ===

--- shock_index ---
  ESI   N válido   Miss%      p5   Mediana      p95      Max
    3     28,745     8.6%   0.406     0.644    1.036    2.686
    4     16,788    16.8%   0.433     0.664    1.107    2.672
    5      2,386    23.1%   0.429     0.661    1.140    2.333

--- comorbidity_count ---
  ESI    Media     Score=0 (%)     Score=1 (%)     Score>=2 (%)    Max
    3     0.89   15,493 (49%)    8,593 (27%)     7,374 (23%)      8
    4     0.52   13,387 (66%)    4,289 (21%)     2,506 (12%)      8
    5     0.49    2,144 (69%)      602 (19%)       356 (11%)      7

--- age_65plus ---
  ESI    N >=65    % >=65   Rate outcome >=65    Rate outcome <65
    3     6,264      19.9%              28.4%               9.8%
    4     1,793       8.9%               8.4%               1.9%
    5       304       9.8%              17.8%               2.0%


In [29]:
# Re-guardar CSV final con datos post-limpieza Harvard + tres features derivadas
# Reemplaza el CSV guardado en la Sección 1 con la versión definitiva

FINAL_COLS = [
    'year', 'esi', 'outcome', 'ADMITHOS', 'TRANOTH', 'DIEDED',
    'AGE', 'SEX', 'ARREMS', 'temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT',
    'PAINSCALE', 'RFV1', 'RFV2', 'shock_index', 'age_65plus',
    'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2',
    'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB', 'comorbidity_count',
    'RACERETH', 'PAYTYPER', 'NUMMED',
]

cols = [c for c in FINAL_COLS if c in df_main.columns]
df_final = df_main[cols].copy()
out_path  = DATA_DIR / 'nhamcs_pooled_2016_2022.csv'
df_final.to_csv(out_path, index=False)

print(f'CSV final: {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas')
print(f'Columnas: {cols}')
print()
print('Verificación features derivadas:')
for feat in ['shock_index', 'comorbidity_count', 'age_65plus']:
    nv = df_final[feat].notna().sum()
    nm = df_final[feat].isna().sum()
    vmin = df_final[feat].min()
    vmax = df_final[feat].max()
    print(f'  {feat:20s}: {nv:,} validos  {nm:,} NaN  min={vmin:.2f}  max={vmax:.2f}  checkOK')


CSV final: 54,744 filas x 36 columnas
Columnas: ['year', 'esi', 'outcome', 'ADMITHOS', 'TRANOTH', 'DIEDED', 'AGE', 'SEX', 'ARREMS', 'temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'PAINSCALE', 'RFV1', 'RFV2', 'shock_index', 'age_65plus', 'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2', 'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB', 'comorbidity_count', 'RACERETH', 'PAYTYPER', 'NUMMED']

Verificación features derivadas:
  shock_index         : 47,919 validos  6,825 NaN  min=0.10  max=2.69  checkOK
  comorbidity_count   : 54,744 validos  0 NaN  min=0.00  max=8.00  checkOK
  age_65plus          : 54,744 validos  0 NaN  min=0.00  max=1.00  checkOK


---
## EDA 14 — Features adicionales derivadas

Se agregan ocho features nuevas al dataset:
- `age_shock_index`: interacción edad × shock index
- Flags de vitales anormales (5): taquicardia, taquipnea, hipotensión, hipoxemia, fiebre
- Flags de missingness informativo (2): presión arterial no registrada, cualquier vital ausente

**Regla NaN → flags:**  
- Flags clínicas (hallazgo fisiológico): si el vital es NaN → flag = 0 (ausencia de dato ≠ presencia del hallazgo).  
- Flags de missingness: si el vital es NaN → flag = 1 (es exactamente lo que capturan).

In [30]:
# Cargar el dataset procesado (resultado de EDA 11-13)
df_feat = pd.read_csv(DATA_DIR / 'nhamcs_pooled_2016_2022.csv')
print(f'Dataset cargado: {df_feat.shape[0]:,} filas x {df_feat.shape[1]} columnas')
print(f'Columnas existentes: {list(df_feat.columns)}')


Dataset cargado: 54,744 filas x 36 columnas
Columnas existentes: ['year', 'esi', 'outcome', 'ADMITHOS', 'TRANOTH', 'DIEDED', 'AGE', 'SEX', 'ARREMS', 'temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'PAINSCALE', 'RFV1', 'RFV2', 'shock_index', 'age_65plus', 'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2', 'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB', 'comorbidity_count', 'RACERETH', 'PAYTYPER', 'NUMMED']


In [31]:
# ── Feature 1: age_shock_index ──────────────────────────────────────────
# El mismo shock index tiene distinto riesgo según la edad del paciente.
# Un adulto mayor compensa peor la taquicardia relativa — conecta con
# la hipótesis del subgrupo ≥65 declarada en el Notebook 00.
# Si AGE o shock_index son NaN → age_shock_index es NaN.
df_feat['age_shock_index'] = df_feat['AGE'] * df_feat['shock_index']

# ── Feature 2: comorbidity_count (verificar existencia) ──────────────────
# Carga crónica acumulada como feature continua (0-12).
# Ya existe en el dataset desde EDA 13 — solo verificamos.
assert 'comorbidity_count' in df_feat.columns, 'comorbidity_count no encontrada'

# ── Feature 3: age_65plus (verificar existencia) ──────────────────────────
# Subgrupo primario confirmatorio del proyecto (declarado en NB 00).
# Ya existe en el dataset desde EDA 13 — solo verificamos.
assert 'age_65plus' in df_feat.columns, 'age_65plus no encontrada'

# ── Features 4-8: Flags de vitales anormales ─────────────────────────────
# Señales fisiológicas binarias interpretables por cualquier triagista.
# Si el vital es NaN → flag = 0 (ausencia de dato ≠ presencia del hallazgo).

# Taquicardia: PULSE > 100 lpm (criterio clínico estándar)
df_feat['tachycardia_flag'] = (df_feat['PULSE'] > 100).astype('Int8')
df_feat.loc[df_feat['PULSE'].isna(), 'tachycardia_flag'] = 0

# Taquipnea: RESPR > 20 rpm (indicador de dificultad respiratoria)
df_feat['tachypnea_flag'] = (df_feat['RESPR'] > 20).astype('Int8')
df_feat.loc[df_feat['RESPR'].isna(), 'tachypnea_flag'] = 0

# Hipotensión: BPSYS < 90 mmHg (criterio de shock según ATLS/protocolo sepsis)
df_feat['hypotension_flag'] = (df_feat['BPSYS'] < 90).astype('Int8')
df_feat.loc[df_feat['BPSYS'].isna(), 'hypotension_flag'] = 0

# Hipoxemia: POPCT < 94% (umbral de intervención en urgencias)
df_feat['hypoxemia_flag'] = (df_feat['POPCT'] < 94).astype('Int8')
df_feat.loc[df_feat['POPCT'].isna(), 'hypoxemia_flag'] = 0

# Fiebre: temp_f > 100.4°F (equivalente a 38°C, criterio SIRS)
df_feat['fever_flag'] = (df_feat['temp_f'] > 100.4).astype('Int8')
df_feat.loc[df_feat['temp_f'].isna(), 'fever_flag'] = 0

# ── Features 9-10: Flags de missingness informativo ──────────────────────
# La ausencia de vitales en ESI 4-5 puede reflejar una decisión operativa
# del triagista: consideró al paciente tan no-urgente que no midió la presión.
# Es señal informativa del contexto de triage, no ruido aleatorio.
# Para estos flags: NaN en la variable base = 1 en la flag.

# bp_missing_flag: presión arterial (BPSYS) no registrada al triage
df_feat['bp_missing_flag'] = df_feat['BPSYS'].isna().astype('Int8')

# any_vital_missing_flag: al menos un vital principal ausente
VITALS_BASE = ['temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT']
df_feat['any_vital_missing_flag'] = (
    df_feat[VITALS_BASE].isna().any(axis=1)
).astype('Int8')

print('Features creadas/verificadas:')
new_feats = ['age_shock_index', 'comorbidity_count', 'age_65plus',
             'tachycardia_flag', 'tachypnea_flag', 'hypotension_flag',
             'hypoxemia_flag', 'fever_flag',
             'bp_missing_flag', 'any_vital_missing_flag']
for f in new_feats:
    nv  = df_feat[f].notna().sum()
    nm  = df_feat[f].isna().sum()
    n1  = int((df_feat[f] == 1).sum()) if df_feat[f].dtype != float else None
    pct1 = n1 / len(df_feat) * 100 if n1 is not None else float('nan')
    print(f'  {f:25s}: {nv:,} validos  NaN={nm:,}  '  
          f'N=1: {n1:,} ({pct1:.1f}%)' if n1 is not None
          else f'  {f:25s}: {nv:,} validos  NaN={nm:,}')


Features creadas/verificadas:
  age_shock_index          : 47,919 validos  NaN=6,825
  comorbidity_count        : 54,744 validos  NaN=0  N=1: 13,484 (24.6%)
  age_65plus               : 54,744 validos  NaN=0  N=1: 8,361 (15.3%)
  tachycardia_flag         : 54,744 validos  NaN=0  N=1: 14,414 (26.3%)
  tachypnea_flag           : 54,744 validos  NaN=0  N=1: 8,739 (16.0%)
  hypotension_flag         : 54,744 validos  NaN=0  N=1: 394 (0.7%)
  hypoxemia_flag           : 54,744 validos  NaN=0  N=1: 2,075 (3.8%)
  fever_flag               : 54,744 validos  NaN=0  N=1: 2,055 (3.8%)
  bp_missing_flag          : 54,744 validos  NaN=0  N=1: 5,605 (10.2%)
  any_vital_missing_flag   : 54,744 validos  NaN=0  N=1: 9,523 (17.4%)


In [32]:
# Distribución de features nuevas por ESI
print('=== DISTRIBUCIÓN DE FEATURES NUEVAS POR ESI ===\n')

# age_shock_index
print('--- age_shock_index (AGE x shock_index) ---')
print(f'{"ESI":>5}  {"N válido":>9}  {"Miss%":>7}  {"p5":>7}  '
      f'{"Mediana":>9}  {"p95":>7}  {"Max":>9}')
for e in [3, 4, 5]:
    sub  = df_feat[df_feat['esi'] == e]['age_shock_index'].dropna()
    miss = df_feat[df_feat['esi'] == e]['age_shock_index'].isna().mean() * 100
    print(f'{e:>5}  {len(sub):>9,}  {miss:>7.1f}%  {sub.quantile(.05):>7.1f}  '
          f'{sub.median():>9.1f}  {sub.quantile(.95):>7.1f}  {sub.max():>9.1f}')

# Flags binarias
FLAGS = ['tachycardia_flag', 'tachypnea_flag', 'hypotension_flag',
         'hypoxemia_flag', 'fever_flag', 'bp_missing_flag', 'any_vital_missing_flag']

print()
print(f'{"Flag":25s}  {"Global%":>8}  {"ESI3%":>7}  {"ESI4%":>7}  {"ESI5%":>7}  '
      f'{"RateOut@flag=1":>15}  {"RateOut@flag=0":>15}')
print('-' * 100)

flag_rows = []
for flag in FLAGS:
    glob_pct = (df_feat[flag] == 1).mean() * 100
    pcts = []
    for e in [3, 4, 5]:
        sub = df_feat[df_feat['esi'] == e]
        pcts.append((sub[flag] == 1).mean() * 100)
    # Tasa de outcome cuando flag=1 vs flag=0
    f1 = df_feat[df_feat[flag] == 1]
    f0 = df_feat[df_feat[flag] == 0]
    r1 = f1['outcome'].mean() * 100 if len(f1) > 0 else 0
    r0 = f0['outcome'].mean() * 100 if len(f0) > 0 else 0
    enr = r1 / r0 if r0 > 0 else float('nan')
    print(f'{flag:25s}  {glob_pct:>8.1f}%  {pcts[0]:>7.1f}%  {pcts[1]:>7.1f}%  '
          f'{pcts[2]:>7.1f}%  {r1:>13.1f}%  {r0:>13.1f}%  (enrich {enr:.2f}x)')
    for e_i, e in enumerate([3, 4, 5]):
        sub = df_feat[df_feat['esi'] == e]
        f1s = sub[sub[flag] == 1]
        f0s = sub[sub[flag] == 0]
        flag_rows.append({
            'flag': flag, 'esi': e,
            'pct_flag1': round(pcts[e_i], 2),
            'outcome_rate_flag1': round(f1s['outcome'].mean() * 100 if len(f1s) > 0 else 0, 2),
            'outcome_rate_flag0': round(f0s['outcome'].mean() * 100 if len(f0s) > 0 else 0, 2),
        })

pd.DataFrame(flag_rows).to_csv(REPORT_DIR / '01_binary_flags_by_esi.csv', index=False)
print('\nGuardado: 01_binary_flags_by_esi.csv')


=== DISTRIBUCIÓN DE FEATURES NUEVAS POR ESI ===

--- age_shock_index (AGE x shock_index) ---
  ESI   N válido    Miss%       p5    Mediana      p95        Max
    3     28,745      8.6%      7.0       25.9     53.6      161.9
    4     16,788     16.8%      4.1       20.3     45.0      102.6
    5      2,386     23.1%      3.2       19.2     45.4       73.2

Flag                        Global%    ESI3%    ESI4%    ESI5%   RateOut@flag=1   RateOut@flag=0
----------------------------------------------------------------------------------------------------
tachycardia_flag               26.3%     25.0%     28.3%     26.8%            9.4%            8.7%  (enrich 1.08x)


tachypnea_flag                 16.0%     13.9%     18.3%     21.4%            9.7%            8.7%  (enrich 1.11x)
hypotension_flag                0.7%      0.6%      0.7%      1.6%           15.2%            8.8%  (enrich 1.73x)
hypoxemia_flag                  3.8%      4.6%      2.4%      4.5%           25.4%            8.2%  (enrich 3.09x)
fever_flag                      3.8%      3.6%      4.2%      2.6%           10.5%            8.8%  (enrich 1.19x)


bp_missing_flag                10.2%      6.6%     14.6%     18.4%            3.7%            9.5%  (enrich 0.39x)


any_vital_missing_flag         17.4%     13.7%     21.5%     28.1%            6.6%            9.3%  (enrich 0.71x)

Guardado: 01_binary_flags_by_esi.csv


In [33]:
# Guardar dataset final con todas las features
FINAL_COLS = [
    # Identificadores y outcome
    'year', 'esi', 'outcome', 'ADMITHOS', 'TRANOTH', 'DIEDED',
    # Set A: triage-only strict
    'AGE', 'SEX', 'ARREMS', 'temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT',
    'PAINSCALE', 'RFV1', 'RFV2',
    # Features derivadas Set A
    'shock_index', 'age_65plus', 'age_shock_index',
    'tachycardia_flag', 'tachypnea_flag', 'hypotension_flag',
    'hypoxemia_flag', 'fever_flag',
    'bp_missing_flag', 'any_vital_missing_flag',
    # Set B: comorbilidades
    'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2',
    'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB',
    # Feature derivada Set B
    'comorbidity_count',
    # Fairness / control
    'RACERETH', 'PAYTYPER',
    # Leakage demo only (Set C — nunca en Set A ni B)
    'NUMMED',
]

cols = [c for c in FINAL_COLS if c in df_feat.columns]
missing_cols = [c for c in FINAL_COLS if c not in df_feat.columns]
if missing_cols:
    print(f'ADVERTENCIA: columnas no encontradas: {missing_cols}')

df_feat[cols].to_csv(DATA_DIR / 'nhamcs_pooled_2016_2022.csv', index=False)

print(f'Dataset final guardado: {len(df_feat):,} filas x {len(cols)} columnas')
print(f'Columnas: {cols}')
print()
print('Resumen de features por set:')
print(f'  Set A (triage-only): AGE, SEX, ARREMS, temp_f, PULSE, RESPR, BPSYS, BPDIAS, POPCT,')
print(f'                       PAINSCALE, RFV1, RFV2, shock_index')
print(f'  Set A derivadas:     age_65plus, age_shock_index, 5 vital flags, 2 missingness flags')
print(f'  Set B extras:        12 comorbilidades + comorbidity_count')
print(f'  Set C (leakage):     NUMMED (solo para demostración, nunca en modelo principal)')


Dataset final guardado: 54,744 filas x 44 columnas
Columnas: ['year', 'esi', 'outcome', 'ADMITHOS', 'TRANOTH', 'DIEDED', 'AGE', 'SEX', 'ARREMS', 'temp_f', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'PAINSCALE', 'RFV1', 'RFV2', 'shock_index', 'age_65plus', 'age_shock_index', 'tachycardia_flag', 'tachypnea_flag', 'hypotension_flag', 'hypoxemia_flag', 'fever_flag', 'bp_missing_flag', 'any_vital_missing_flag', 'ASTHMA', 'CANCER', 'COPD', 'CHF', 'DEPRN', 'DIABTYP1', 'DIABTYP2', 'ESRD', 'EDHIV', 'HTN', 'OBESITY', 'SUBSTAB', 'comorbidity_count', 'RACERETH', 'PAYTYPER', 'NUMMED']

Resumen de features por set:
  Set A (triage-only): AGE, SEX, ARREMS, temp_f, PULSE, RESPR, BPSYS, BPDIAS, POPCT,
                       PAINSCALE, RFV1, RFV2, shock_index
  Set A derivadas:     age_65plus, age_shock_index, 5 vital flags, 2 missingness flags
  Set B extras:        12 comorbilidades + comorbidity_count
  Set C (leakage):     NUMMED (solo para demostración, nunca en modelo principal)


---
## Plan de explicabilidad — SHAP (a implementar en NB02)

### Objetivo clínico

El sistema no genera alertas binarias — genera **alertas explicativas**. Para cada paciente alertado, el sistema produce:

> *"Este paciente tiene alta probabilidad de requerir hospitalización o traslado. Los factores que más contribuyen son: [factor 1], [factor 2], [factor 3]."*

Formato preliminar ilustrativo; la implementación final está en NB05.

**Por qué esto es clínicamente necesario:**
- Una alerta sin explicación no tiene validez operativa: el triagista no puede validarla
- El triagista debe poder confirmar si la alerta tiene sentido dado lo que acaba de ver
- La explicabilidad no es un add-on — es parte del diseño del sistema de soporte
- Reduce el riesgo de alarm fatigue: alertas explicadas son más creíbles

---

### Plan de implementación (Notebook 02)

**1. SHAP TreeExplainer sobre el modelo LightGBM final**
```python
import shap
explainer = shap.TreeExplainer(model_lgbm)
shap_values = explainer.shap_values(X_val)
```

**2. SHAP values por paciente en el set de validación**  
Para cada fila: vector de contribuciones de cada feature al score de riesgo.

**3. Para pacientes alertados: top 3 features con mayor contribución positiva**  
Se ordenan los SHAP values del paciente y se toman los 3 más altos (contribución positiva al riesgo).

**4. Generación de texto de alerta clínica automático**  
Usando el mapeo de nombres de variables a términos clínicos:

| Variable | Texto clínico legible |
|----------|----------------------|
| `AGE` | edad avanzada |
| `shock_index` | índice de shock elevado |
| `age_shock_index` | riesgo amplificado por edad y estado hemodinámico |
| `CHF` | insuficiencia cardíaca congestiva |
| `COPD` | enfermedad pulmonar obstructiva crónica |
| `CANCER` | antecedente de cáncer |
| `ESRD` | insuficiencia renal crónica severa |
| `comorbidity_count` | alta carga de comorbilidades |
| `tachycardia_flag` | taquicardia al ingreso |
| `tachypnea_flag` | taquipnea al ingreso |
| `hypotension_flag` | hipotensión al ingreso |
| `hypoxemia_flag` | hipoxemia al ingreso (SpO2 < 94%) |
| `fever_flag` | fiebre al ingreso |
| `bp_missing_flag` | presión arterial no registrada al triage |
| `any_vital_missing_flag` | vitales incompletos al triage |
| `PAINSCALE` | dolor intenso reportado |
| `ARREMS` | llegada en ambulancia |
| `RFV1` | motivo de consulta de alto riesgo |
| `age_65plus` | paciente mayor de 65 años |

**5. Ejemplos reales de alertas en el set de validación**  
Mostrar 3-5 casos de pacientes alertados con su texto clínico generado,  
y si el outcome real fue positivo o negativo (para evaluar la coherencia).

**6. SHAP summary plot global**  
Beeswarm plot con todas las features — muestra qué variables impulsan el modelo  
y en qué dirección. Se genera por ESI separado.

**7. SHAP dependence plots**  
Para las 3 features más importantes por ESI: relación entre valor de la feature  
y contribución al score, con interacción de color por una segunda feature.

---

### Prerequisito para implementar en NB02

```bash
pip install shap
```

El modelo de referencia para SHAP es el **Set A** (triage-only strict),  
porque es el que tiene validez operativa real (sin supuestos de acceso a historia clínica).  
Se muestra adicionalmente el SHAP del Set B si mejora la explicabilidad clínica.